##Persiapan Praktikum


In [ ]:
!pip install faker -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = '/content/data'
DIR_SIMPAN = '/content/drive/MyDrive/BigData/Praktikum2'
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[]


##K-1. Import Library dan Inisialisasi
Dilakukan import terhadap library yang akan digunakan dalam praktikum, yaitu numpy, pandas, faker, dan random.

In [ ]:
import numpy as np
import pandas as pd
from faker import Faker
import random


##K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)
Kode program pada cell ini berfungsi untuk menghasilkan 500 baris data + 15 data duplikat transaksi fiktif berbahasa Indonesia menggunakan Faker dengan menyisipkan berbagai anomali sejak awal iterasi, seperti ketidakseragaman format tanggal dan harga, inkonsistensi teks, serta nilai kosong bawaan. Setelah dimuat ke dalam Pandas DataFrame, data tersebut secara sengaja disuntikkan lagi dengan missing values (np.nan) pada beberapa kolom, ditambahkan 15 baris duplikat, dan diacak urutannya sebelum akhirnya disimpan menjadi berkas transaksi_mentah.csv

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


##K-3. Deteksi dan Penanganan Missing Value
Dilakukan pemeriksaan keterisian data melalui df.isnull() yang mengubah setiap sel menjadi nilai boolean bernilai True untuk sel kosong, lalu hasilnya dijumlahkan per kolom melalui sum() sehingga dicetak Series berisi nama setiap kolom beserta banyaknya nilai kosong pada kolom tersebut.

In [ ]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


##K-4. Deteksi dan Penanganan Duplicate
Dicetak jumlah baris duplikat sempurna melalui df.duplicated().sum() yang menandai baris kedua dan seterusnya apabila seluruh nilai kolomnya sama persis, lalu dicetak pula jumlah identitas transaksi yang berulang melalui df['transaction_id'].duplicated().sum() sebagai pemeriksaan tambahan pada kolom yang seharusnya unik.
Baris duplikat dibuang melalui df.drop_duplicates() yang mempertahankan kemunculan pertama setiap baris, hasilnya ditugaskan kembali ke df, lalu jumlah baris yang tersisa dicetak melalui len(df) sebagai pembanding terhadap jumlah baris sebelum penghapusan. Setelah program dijalankan, maka dapat diketahui bahwa seluruh terdapat 15 data duplikat pada seluruh dataset. Setelah data duplikat dihapus, tersisa 500 baris pada dataset


In [ ]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate: ", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates(): ", len(df))

Baris duplicate (semua kolom sama): 15
transaction_id duplicate:  15
Jumlah baris setelah drop_duplicates():  500


##K-3. Deteksi dan Penanganan Missing Value
Proses pembersihan missing values dilakukan dengan menghapus baris yang kosong secara spesifik pada kolom customer_name dan payment_method menggunakan dropna(), serta mengisi nilai null pada shipping_city dengan label "Tidak Diketahui" melalui fillna(). Setelah penanganan tersebut, dataset akhirnya menyisakan 490 baris data siap pakai, sementara kolom rating yang memiliki jumlah nilai kosong terbanyak tetap dipertahankan

#Alasan Langkah K-3 Dilakukan 2 kali, sebelum dan setelah K-4 agar jumlah baris pada dataset setelah drop duplicates yaitu 500, mengikuti output yang terdapat pada modul praktikum

In [ ]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_dropna(): ", len(df))

Jumlah baris setelah drop_dropna():  490


##K-5. Koreksi Tipe Data dan Standardisasi Format
a. Standardisasi teks kategorikal (category, payment_method, shipping_city):
Kolom teks category, payment_method, dan shipping_city distandardisasi dengan menghapus spasi berlebih dan menyeragamkan huruf awal menjadi kapital menggunakan fungsi strip() dan title(). Khusus pada kolom payment_method, nilai "Cod" dikoreksi kembali menjadi "COD" melalui fungsi replace() agar penulisan singkatan tersebut tetap sesuai dengan aslinya

In [ ]:
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

##K-5. Koreksi Tipe Data dan Standardisasi Format
b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)

Proses penyeragaman format pada kolom price dilakukan dengan mendefinisikan sebuah fungsi khusus bernama bersihkan_harga(x). Fungsi ini bekerja dengan terlebih dahulu mengecek nilai kosong menggunakan pd.isna() agar sel yang null dapat dilewati dan langsung dikembalikan sebagai np.nan. Jika berisi data, nilai tersebut diubah menjadi string untuk dibersihkan menggunakan fungsi strip() guna membuang spasi di kedua tepinya, dilanjutkan dengan tiga tahap replace() berurutan untuk menghapus awalan "Rp", menghilangkan titik pemisah ribuan, dan mengubah koma menjadi titik agar sesuai dengan standar format desimal pada Python.

Setelah teks berhasil dibersihkan, nilai tersebut dikonversi menjadi bilangan pecahan (float) di dalam blok try-except. Penggunaan metode penanganan galat ini memastikan bahwa jika ada nilai yang tidak valid dan memicu ValueError, sistem akan dengan aman mengubahnya menjadi np.nan tanpa menghentikan eksekusi keseluruhan program. Terakhir, fungsi ini diterapkan ke seluruh baris pada kolom price menggunakan metode apply(), sehingga keempat variasi penulisan harga yang berantakan kini menyatu menjadi satu kolom bertipe numerik yang seragam.

In [ ]:
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

##K-5. Koreksi Tipe Data dan Standardisasi Format
c. Standardisasi format tanggal ke YYYY-MM-DD:

Penyeragaman format pada kolom transaction_date dilakukan dengan membuat fungsi parse_tanggal(x) yang mengiterasi tiga pola penulisan tanggal, yaitu %Y-%m-%d, %d/%m/%Y, dan %d-%m-%Y. Melalui blok try-except, setiap pola diuji coba untuk mengonversi nilai menjadi objek tanggal menggunakan pd.to_datetime() dengan parameter format secara eksplisit. Penggunaan parameter ini sangat penting guna mencegah sistem keliru menukar posisi angka hari dan bulan. Jika semua pola gagal dicocokkan, fungsi secara otomatis akan mengembalikan pd.NaT (Not a Time) sebagai penanda data yang tidak sah tanpa menghentikan eksekusi program.

Setelah fungsi tersebut diterapkan pada kolom transaction_date menggunakan apply(), seluruh objek tanggal yang berhasil diekstrak kemudian diseragamkan kembali ke dalam satu format baku (YYYY-MM-DD) melalui metode .dt.strftime("%Y-%m-%d"). Hasil akhir ini ditugaskan kembali ke kolom yang sama, sehingga variasi penulisan tanggal yang sebelumnya berantakan kini menjadi seragam, konsisten, dan siap untuk dianalisis.

In [ ]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

##K-5. Koreksi Tipe Data dan Standardisasi Format
d. Finalisasi tipe data:

Dikoreksi tipe data kedua kolom numerik, yaitu kolom quantity dikonversi ke bilangan bulat melalui astype(int) dan kolom price dikonversi ke bilangan pecahan melalui astype(float), lalu masing-masing hasilnya di-assign kembali ke kolom yang sama.

In [ ]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

##K-6. Ekspor Dataset Bersih
DataFrame di-export menjadi file transaksi_bersih.csv melalui to_csv() berargumen index=False, sehingga indeks tidak ikut tertulis sebagai kolom. Jumlah barisnya kemudian dicetak melalui len(df).

In [ ]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan: ", len(df), "baris")

Dataset bersih tersimpan:  490 baris


##K-6. Ekspor Dataset Bersih
Tahap terakhir melibatkan penggunaan modul shutil dan pustaka pandas untuk memastikan file keluaran, yaitu transaksi_mentah.csv dan transaksi_bersih.csv, tersimpan di lokasi yang tepat. Program akan mengiterasi kedua nama file tersebut dan secara cerdas mencari keberadaannya melalui tiga kemungkinan direktori kerja. Jika sebuah file gagal ditemukan, sistem akan memunculkan peringatan [GAGAL] beserta anjuran bagi pengguna untuk menjalankan ulang proses pembangkitan data, lalu secara otomatis beralih ke file berikutnya.

Sebaliknya, jika file terdeteksi, program akan langsung menyalinnya ke direktori penyimpanan (DIR_SIMPAN) menggunakan fungsi shutil.copy(). Sebagai langkah verifikasi ekstra, file yang baru disalin tersebut dibaca kembali menggunakan Pandas untuk dihitung jumlah barisnya, kemudian sistem akan mencetak status [OK] lengkap dengan rincian jalur tujuan dan total baris data. Eksekusi ini diakhiri dengan menampilkan isi dari direktori penyimpanan, di mana dapat dipastikan bahwa dataset transaksi_bersih.csv telah berhasil diamankan dan berisi 490 baris data yang siap untuk dianalisis lebih lanjut.

In [ ]:
import shutil
import pandas as pd

FILES = ["transaksi_mentah.csv", "transaksi_bersih.csv"]

for nama in FILES:
    kandidat = [os.path.join(DIR_KERJA, nama), os.path.join("/content", nama), nama]
    asal = next((p for p in kandidat if os.path.exists(p)), None)

    if asal is None:
        print(f"[GAGAL] {nama} tidak ditemukan — jalankan ulang sel K-2 / K-6.")
        continue

    tujuan = os.path.join(DIR_SIMPAN, nama)
    shutil.copy(asal, tujuan)
    jumlah = len(pd.read_csv(tujuan))
    print(f"[OK] {nama} -> {tujuan} ({jumlah} baris)")

print("\nIsi folder Praktikum2:", os.listdir(DIR_SIMPAN))

[OK] transaksi_mentah.csv -> /content/drive/MyDrive/BigData/Praktikum2/transaksi_mentah.csv (515 baris)
[OK] transaksi_bersih.csv -> /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih.csv (490 baris)

Isi folder Praktikum2: ['transaksi_mentah.csv', 'transaksi_bersih.csv']



##STUDI KASUS
Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan bulanan: total transaksi yang
dilaporkan tim IT (515) tidak sama dengan total yang dipakai tim Finance (490). Sebagai calon data
engineer, jelaskan kepada tim Finance:

##1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda lakukan:
Kedua angka berbeda karena keduanya menghitung data pada tahap yang berbeda di dalam satu alur yang sama. Angka 515 adalah jumlah baris pada transaksi_mentah.csv, yaitu hasil data acquisition sebelum diproses, sedangkan angka 490 adalah jumlah baris pada transaksi_bersih.csv, yaitu hasil setelah preprocessing.

Operasi pertama adalah penghapusan baris duplikat dengan drop_duplicates(). Pemeriksaan df.duplicated() mencatat adanya baris yang identik pada seluruh kolom, termasuk transaction_id yang berulang, padahal satu transaksi seharusnya hanya tercatat satu kali. Baris seperti ini akan menggandakan nilai penjualan apabila dijumlahkan, sehingga harus dibuang lebih dahulu.

Operasi kedua adalah penanganan missing value pada kolom kunci. Baris yang tidak memiliki customer_name dan payment_method dibuang karena transaksi tanpa identitas pelanggan dan tanpa metode pembayaran tidak dapat diverifikasi ke sumbernya. Sebaliknya, kolom shipping_city yang kosong  tidak dibuang melainkan diisi "Tidak diketahui", karena kota pengiriman tidak memengaruhi keabsahan transaksi. Jumlah baris yang dibuang tidak sama dengan penjumlahan angka-angka tersebut karena sebagian baris terkena lebih dari satu kriteria sekaligus.

Dengan demikian, angka 515 menjawab pertanyaan "berapa banyak catatan yang masuk ke sistem", sedangkan angka 490 menjawab "berapa banyak catatan yang layak dipakai untuk pelaporan". Perbedaan keduanya tidak menunjukkan kesalahan salah satu tim, tetapi menunjukkan bahwa kedua tim belum menyepakati pada tahap mana angka penjualan resmi diambil.




##2. Apakah 490 baris "lebih benar" dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity:
Angka 490 tidak "lebih benar" secara mutlak, tetapi lebih layak dipercaya untuk keperluan pelaporan penjualan. Veracity pada 5V berbicara tentang kelayakan data untuk dipercaya.

Untuk pertanyaan tentang nilai penjualan, 490 baris lebih tepat karena 515 baris justru mengandung cacat yang menggeser hasil ke atas. Baris duplikat yang terdeteksi oleh df.duplicated() memuat transaction_id yang sama, sehingga satu transaksi yang sama akan dihitung dua kali apabila nilainya dijumlahkan, dan total penjualan menjadi lebih besar daripada yang sebenarnya terjadi.
Sebaliknya, untuk pertanyaan tentang kualitas proses pencatatan, 515 baris justru yang benar, karena angka itulah rekaman apa adanya dari sistem sumber.



##3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing value kepada tim Finance yang ingin tahu "rating rata-rata semua transaksi":
Pertama-tama perlu diluruskan bahwa rating yang kosong bukan berarti pelanggan memberi nilai rendah, melainkan pelanggan tidak memberi nilai sama sekali. Pada transaksi_mentah.csv terdapat 166 dari 515 baris yang kolom rating-nya kosong, sehingga pertanyaan "rating rata-rata semua transaksi" sebenarnya tidak dapat dijawab.

Kolom tersebut sengaja tidak diisi (di-impute) karena setiap cara pengisian akan memasukkan angka karangan ke dalam data. Mengisinya dengan 0 akan menurunkan rata-rata secara palsu, padahal 0 bukan nilai yang mungkin diberikan pelanggan pada skala 1 sampai 5. Mengisinya dengan nilai rata-rata atau modus akan membuat sekitar sepertiga baris memiliki nilai yang sama persis, sehingga sebaran rating menyempit dan selisih antarkategori tampak lebih kecil daripada kenyataannya. Berbeda dengan shipping_city yang diisi "Tidak diketahui" karena hanya berfungsi sebagai keterangan, rating adalah angka yang langsung ikut dihitung, sehingga kesalahan pengisiannya merambat ke setiap hasil analisis di hilir.

